In [ ]:
# ============================================================
# CELL 1 — Setup, Imports, Dataset Path, DataFrame + Split
# (Kaggle version — dataset is mounted via Add Input, not downloaded)
# ============================================================

!pip install -q albumentations opencv-python-headless torchsummary

import os
import io
import glob
import base64
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Dataset is mounted under /kaggle/input — on this notebook Kaggle nested it
# one level deeper (/kaggle/input/datasets/...) instead of the classic
# /kaggle/input/<dataset-slug>/ path, so we point at the top-level folder
# and let the recursive glob below find the files wherever they landed.
dataset_path = "/kaggle/input"
print(f"Dataset path: {dataset_path}")
print(os.listdir(dataset_path))

metadata_path = glob.glob(os.path.join(dataset_path, "**", "HAM10000_metadata.csv"), recursive=True)[0]
meta_df = pd.read_csv(metadata_path)

image_paths = glob.glob(os.path.join(dataset_path, "**", "*.jpg"), recursive=True)
id_to_path = {os.path.splitext(os.path.basename(p))[0]: p for p in image_paths}
meta_df['path'] = meta_df['image_id'].map(id_to_path)
meta_df = meta_df.dropna(subset=['path']).reset_index(drop=True)
print(f"Total labeled images found: {len(meta_df)}")

CLASSES = sorted(meta_df['dx'].unique())
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
meta_df['label'] = meta_df['dx'].map(class_to_idx)
print(f"Classes ({len(CLASSES)}): {CLASSES}")

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(gss.split(meta_df, groups=meta_df['lesion_id']))
train_df = meta_df.iloc[train_idx].reset_index(drop=True)
val_df = meta_df.iloc[val_idx].reset_index(drop=True)

print(f"Train samples: {len(train_df)} | Val samples: {len(val_df)}")
print("Train class distribution:\n", train_df['dx'].value_counts())
print("Val class distribution:\n", val_df['dx'].value_counts())

MALIGNANT_CLASSES = {'akiec', 'bcc', 'mel'}
malignant_idx = [i for i, c in enumerate(CLASSES) if c in MALIGNANT_CLASSES]
print(f"Malignant class indices: {malignant_idx} -> {[CLASSES[i] for i in malignant_idx]}")

In [ ]:
# ============================================================
# CELL 2 — Dataset, Transforms, DataLoaders
# ============================================================

class HAMDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image = Image.open(self.df.loc[idx, 'path']).convert('RGB')
        label = torch.tensor(self.df.loc[idx, 'label'], dtype=torch.long)
        if self.transform:
            image = self.transform(image)
        return image, label

IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02),
    transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.08)),
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

BATCH_SIZE = 32
train_loader = DataLoader(HAMDataset(train_df, train_transforms), batch_size=BATCH_SIZE,
                           shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(HAMDataset(val_df, val_transforms), batch_size=BATCH_SIZE,
                         shuffle=False, num_workers=2, pin_memory=torch.cuda.is_available())

In [ ]:
# ============================================================
# CELL 3 — Focal Loss & EfficientNet Model (with Grad-CAM hooks)
# ============================================================

class MultiClassFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        return (((1.0 - pt) ** self.gamma) * ce_loss).mean()

# Softer reweighting: sqrt of inverse frequency instead of full linear
# 'balanced' weighting. Full balanced weighting + Focal Loss's own gamma
# term double-compensates for imbalance and overcorrects (this is what
# caused the "everything is malignant" collapse last run).
raw_counts = train_df['label'].value_counts().sort_index().values
sqrt_inv_freq = 1.0 / np.sqrt(raw_counts)
alpha_weights = torch.tensor(
    sqrt_inv_freq / sqrt_inv_freq.sum() * len(CLASSES), dtype=torch.float32
).to(device)
criterion = MultiClassFocalLoss(alpha=alpha_weights, gamma=1.5)

class DermaEfficientNet(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()
        self.backbone = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        in_features = self.backbone.classifier[1].in_features
        self.backbone.classifier = nn.Sequential(nn.Dropout(p=0.3, inplace=True), nn.Linear(in_features, num_classes))
        self.gradients, self.activations, self._hook = None, None, None

    def forward(self, x):
        x = self.backbone.features(x)
        self.activations = x
        if x.requires_grad:
            if self._hook is not None:
                self._hook.remove()
            self._hook = x.register_hook(lambda g: setattr(self, 'gradients', g))
        x = self.backbone.avgpool(x)
        x = torch.flatten(x, 1)
        return self.backbone.classifier(x)

model = DermaEfficientNet(num_classes=len(CLASSES)).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

def malignant_recall(y_true, y_pred):
    return recall_score(y_true, y_pred, labels=malignant_idx, average='macro', zero_division=0)

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)

# New: F1 restricted to the malignant classes only. Unlike recall alone,
# this can't be gamed by over-predicting malignant everywhere — it also
# punishes the false positives that tanked precision last run.
def malignant_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, labels=malignant_idx, average='macro', zero_division=0)

In [ ]:
# ============================================================
# CELL 4 — Initial Training (frozen backbone)
# Checkpoints on malignant-class F1 — balances catching cancer
# against not crying wolf on every borderline lesion.
# ============================================================

EPOCHS = 15
best_score = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += torch.sum(torch.argmax(outputs, 1) == labels.data)
        total += labels.size(0)

    model.eval()
    val_correct, val_total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, 1)
            val_correct += torch.sum(preds == labels.data)
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels_arr = np.array(all_labels)
    val_acc = (val_correct.double() / val_total).item()
    f1 = macro_f1(all_labels_arr, all_preds)
    mal_f1 = malignant_f1(all_labels_arr, all_preds)
    mal_rec = malignant_recall(all_labels_arr, all_preds)
    scheduler.step(mal_f1)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {running_loss/total:.4f} Acc: {(correct.double()/total).item()*100:.2f}% "
          f"| Val Acc: {val_acc*100:.2f}% | Macro F1: {f1*100:.2f}% | Malignant F1: {mal_f1*100:.2f}% | Malignant Recall: {mal_rec*100:.2f}%")

    if mal_f1 > best_score:
        best_score = mal_f1
        torch.save(model.state_dict(), "best_dermascan_efficientnet.pth")
        print(f"--> Saved new best checkpoint (malignant F1: {mal_f1*100:.2f}%)")

In [ ]:
# ============================================================
# CELL 5 — Fine-Tuning Stage (full backbone unfrozen)
# best_score is NOT reset — only saves if it beats Cell 4's best.
# ============================================================

for param in model.parameters():
    param.requires_grad = True

fine_tune_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
fine_tune_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(fine_tune_optimizer, T_max=15, eta_min=1e-6)
fine_tune_criterion = MultiClassFocalLoss(alpha=alpha_weights, gamma=1.5)

FINE_TUNE_EPOCHS = 15
print("Starting Fine-Tuning Stage...")

for epoch in range(FINE_TUNE_EPOCHS):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        fine_tune_optimizer.zero_grad()
        outputs = model(images)
        loss = fine_tune_criterion(outputs, labels)
        loss.backward()
        fine_tune_optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += torch.sum(torch.argmax(outputs, 1) == labels.data)
        total += labels.size(0)
    fine_tune_scheduler.step()

    model.eval()
    val_correct, val_total = 0, 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, 1)
            val_correct += torch.sum(preds == labels.data)
            val_total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_labels_arr = np.array(all_labels)
    val_acc = (val_correct.double() / val_total).item()
    f1 = macro_f1(all_labels_arr, all_preds)
    mal_f1 = malignant_f1(all_labels_arr, all_preds)
    mal_rec = malignant_recall(all_labels_arr, all_preds)

    print(f"Fine-Tune Epoch [{epoch+1}/{FINE_TUNE_EPOCHS}] | Train Acc: {(correct.double()/total).item()*100:.2f}% "
          f"| Val Acc: {val_acc*100:.2f}% | Macro F1: {f1*100:.2f}% | Malignant F1: {mal_f1*100:.2f}% | Malignant Recall: {mal_rec*100:.2f}%")

    if mal_f1 > best_score:
        best_score = mal_f1
        torch.save(model.state_dict(), "best_dermascan_efficientnet.pth")
        print(f"--> Saved improved checkpoint (malignant F1: {mal_f1*100:.2f}%)")

In [ ]:
model.load_state_dict(torch.load("best_dermascan_efficientnet.pth", map_location=device))
model.eval()

y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(torch.argmax(probs, 1).cpu().numpy())
        y_probs.extend(probs.cpu().numpy())
y_true, y_pred, y_probs = np.array(y_true), np.array(y_pred), np.array(y_probs)

print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print(f"Malignant F1: {malignant_f1(y_true, y_pred)*100:.2f}%")
print(f"Malignant Recall: {malignant_recall(y_true, y_pred)*100:.2f}%")

In [ ]:
# ============================================================
# CELL 6 — Load Best Checkpoint & Full Evaluation
# ============================================================

model.load_state_dict(torch.load("best_dermascan_efficientnet.pth", map_location=device))
model.eval()

y_true, y_pred, y_probs = [], [], []
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(torch.argmax(probs, 1).cpu().numpy())
        y_probs.extend(probs.cpu().numpy())
y_true, y_pred, y_probs = np.array(y_true), np.array(y_pred), np.array(y_probs)

print("="*65 + "\n                   DERMASCAN CLINICAL REPORT\n" + "="*65)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))
print(f"Overall Macro ROC-AUC Score: {roc_auc_score(y_true, y_probs, multi_class='ovr', average='macro'):.4f}")
print(f"Macro F1: {macro_f1(y_true, y_pred)*100:.2f}%")
print(f"Malignant-Class Macro F1 (checkpoint criterion): {malignant_f1(y_true, y_pred)*100:.2f}%")
print(f"Malignant-Class Macro Recall (akiec/bcc/mel): {malignant_recall(y_true, y_pred)*100:.2f}%")
print("="*65)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', xticklabels=CLASSES, yticklabels=CLASSES)
plt.xlabel('Predicted Label'); plt.ylabel('True Label')
plt.title('Clinical Lesion Confusion Matrix')
plt.tight_layout(); plt.show()

RISK_TIER = {'nv': 'Low', 'bkl': 'Low', 'df': 'Low', 'vasc': 'Low', 'akiec': 'Moderate', 'bcc': 'High', 'mel': 'High'}

In [ ]:
# ============================================================
# CELL 7 — Grad-CAM Visualization
# ============================================================

def generate_gradcam(model, image_path, target_class_idx=None):
    model.eval()
    raw_img = Image.open(image_path).convert('RGB')
    cv_img = cv2.resize(cv2.cvtColor(np.array(raw_img), cv2.COLOR_RGB2BGR), (224, 224))

    input_tensor = val_transforms(raw_img).unsqueeze(0).to(device)
    input_tensor.requires_grad_(True)
    model.zero_grad()
    output = model(input_tensor)
    probs = F.softmax(output, dim=1).detach().cpu().numpy()[0]
    if target_class_idx is None:
        target_class_idx = int(np.argmax(probs))
    output[0, target_class_idx].backward()

    weights = np.mean(model.gradients[0].cpu().numpy(), axis=(1, 2))
    activations = model.activations[0].detach().cpu().numpy()
    cam = np.maximum(np.tensordot(weights, activations, axes=([0], [0])), 0)
    cam = cv2.resize(cam, (224, 224))
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    overlay = cv2.cvtColor(cv2.addWeighted(cv_img, 0.6, heatmap, 0.4, 0), cv2.COLOR_BGR2RGB)
    pred_class = CLASSES[target_class_idx]
    return raw_img, overlay, pred_class, probs[target_class_idx], RISK_TIER.get(pred_class, 'Unknown')

sample_row = val_df[val_df['dx'] == 'mel'].iloc[0]
raw, cam_overlay, pred_cls, conf, tier = generate_gradcam(model, sample_row['path'])

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(raw); axes[0].set_title(f"Ground Truth: {sample_row['dx']}"); axes[0].axis('off')
axes[1].imshow(cam_overlay); axes[1].set_title(f"Pred: {pred_cls} ({conf*100:.1f}%) — Risk: {tier}"); axes[1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# CELL 8 — Save Trained Weights for Download
# ============================================================

# On Kaggle, files saved to /kaggle/working/ show up in the notebook's
# "Output" tab (right panel) after you commit/save the notebook —
# no special download call needed like Colab's files.download().
import shutil
shutil.copy("best_dermascan_efficientnet.pth", "/kaggle/working/best_dermascan_efficientnet.pth")
print("Checkpoint saved to /kaggle/working/. Click 'Save Version' (top right) "
      "to commit the run, then grab the file from the Output tab.")